# Spark on Kubernetes: Cloud-Native Pod-Based Architecture

**Module**: 12_advanced_topics  
**Notebook**: 02_spark_on_kubernetes.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Apache Spark runs natively on Kubernetes: Driver pods, Executor pods, Spark Operator, and cloud-native scaling.
- **Why does this topic exist?**: Kubernetes is rapidly becoming the modern standard for cloud-native big data infrastructure.
- **What problem does it solve?**: Managing ephemeral pod lifecycles and Docker container images.
- **Where is it used in real-world Data Engineering?**: Running Spark on EKS, GKE, and AKS.

---

## 2. Spark on Kubernetes Architecture
```
Kubernetes Control Plane (kube-apiserver):
  1. spark-submit --master k8s://https://k8s-apiserver:443
  2. K8s launches Driver Pod.
  3. Driver Pod requests Executor Pods directly from kube-apiserver.
  4. K8s schedules Executor Pods on available K8s nodes.
  5. Executor Pods connect back to Driver Pod via K8s headless service.
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("K8s_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("K8s submission equivalent command:")
print("""
spark-submit \
  --master k8s://https://kubernetes.default.svc \
  --deploy-mode cluster \
  --name spark-k8s-job \
  --conf spark.kubernetes.container.image=spark-local:4.2.1 \
  --conf spark.kubernetes.driver.pod.name=spark-driver-pod \
  --conf spark.executor.instances=4 \
  local:///opt/spark-apps/sales_report.py
""")

---

## Experiment: Comparing K8s vs Standalone

- **1. What are we testing?**: Contrast K8s pod lifecycle with Docker Compose standalone.
- **2. What do we expect?**: K8s creates ephemeral pods on demand; Standalone uses persistent worker daemons.

In [ ]:
print('K8s provides fine-grained container isolation and auto-scaling via Cluster Autoscaler.')

**3. What happened?**: Explained K8s pod dynamics.  
**4. Why did it happen?**: K8s provisions dedicated containers per executor.  
**5. Practical Engineering Lesson**: K8s eliminates idle worker costs through cloud autoscaling.

---

## Practical Exercises

### Exercise 1 (Beginner): K8s Master URL Syntax
**Problem Statement**: What is the syntax for Spark K8s master?  
**Input / Context**: URL

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Spark Operator Concept
**Problem Statement**: Explain Spark on K8s Operator.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Driver Headless Service
**Problem Statement**: Why K8s requires a headless service for Driver.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (K8s Master URL Syntax):
```python
print('k8s://https://<k8s-apiserver-host>:<port>')
```

#### Solution 2 (Spark Operator Concept):
```python
print('CRD (Custom Resource Definition) allowing declarative SparkApplication YAML manifests.')
```

#### Solution 3 (Driver Headless Service):
```python
print('Enables executor pods to route Netty RPC traffic back to the dynamic driver pod IP.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Cloud-Native**: Pods as executors.
- **Autoscaling**: Integrates with K8s cluster autoscaler.

### Essential Technical Terms
- `Driver Pod`: Pod hosting driver process.
- `Spark Operator`: Kubernetes CRD controller for Spark.

### Common Engineering Mistakes
- **Missing ServiceAccount RBAC permissions**: Driver pod fails to launch executor pods if K8s RBAC is missing.

### Interview Questions on This Topic
- **Q: How does the Driver communicate with executors on Kubernetes?**
  - *A*: Through a Kubernetes headless service that routes Netty RPC packets directly to executor pod IPs.

### What to Study Next
Open 03_cluster_vs_client_mode.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')